# 🎯 [K14_M02] 데이터 표 정리하고 점검하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K14/k14_m02_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·부대관리 (Level 1) / K14 군 생활 데이터 분석 |
| **미션** | 미션 02. 데이터 표 정리하고 점검하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 군 생활 데이터 분석, HITL 수기 검산, 시각화 왜곡 방지 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [Colab 환경 설정 및 헬퍼 함수] 데이터 분석, 가상 데이터 로더 & 마크다운 렌더러
# ============================================================================
!pip install -q pandas numpy tabulate matplotlib seaborn

import pandas as pd
import numpy as np
import json
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# 폰트 설정 (Colab / Linux 호환)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ----------------------------------------------------------------------------
# [데이터셋 자동 로드] 가상 한빛대대 4주간 생활관 생활 데이터 (12행)
# ----------------------------------------------------------------------------
data_records = [
    {'주차': '1주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 112, '싸지방': 54, 'PX': 140, '수면만족': 3.8, '식단만족': 3.9, '건의': 3},
    {'주차': '1주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 98, '싸지방': 31, 'PX': 160, '수면만족': 3.6, '식단만족': 3.5, '건의': 4},
    {'주차': '1주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 104, '싸지방': 47, 'PX': 118, '수면만족': 3.7, '식단만족': 3.2, '건의': 5},
    {'주차': '2주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 118, '싸지방': 58, 'PX': 138, '수면만족': 3.8, '식단만족': 3.8, '건의': 2},
    {'주차': '2주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 92, '싸지방': 29, 'PX': 166, '수면만족': 3.4, '식단만족': 3.4, '건의': 5},
    {'주차': '2주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 106, '싸지방': 50, 'PX': 121, '수면만족': 3.6, '식단만족': 3.1, '건의': 7},
    {'주차': '3주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 121, '싸지방': 61, 'PX': 142, '수면만족': 3.7, '식단만족': 4.0, '건의': 4},
    {'주차': '3주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 85, '싸지방': 26, 'PX': 171, '수면만족': 3.2, '식단만족': 3.4, '건의': 7},
    {'주차': '3주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 103, '싸지방': 49, 'PX': 117, '수면만족': 3.7, '식단만족': 3.3, '건의': 9},
    {'주차': '4주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 125, '싸지방': 66, 'PX': 135, '수면만족': 3.7, '식단만족': 4.1, '건의': 3},
    {'주차': '4주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 80, '싸지방': 24, 'PX': 178, '수면만족': 3.0, '식단만족': 3.3, '건의': 8},
    {'주차': '4주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 108, '싸지방': 52, 'PX': 120, '수면만족': 3.7, '식단만족': 3.2, '건의': 12}
]

df = pd.DataFrame(data_records)

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드합니다."""
    filename = f"k14_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K14", "../uploads/K14"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 분석 결과 및 검증 리포트를 시각적으로 렌더링합니다."""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 분석 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ [K14] 실습 환경, 12행 생활관 데이터셋(df) 및 시각화 렌더러 준비 완료!")
display(df)



---

## 🎯 미션 02. 데이터 표 정리하고 점검하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **AI에 넣을 데이터를 표로 정리하고 이상한 값을 점검합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **12행(4주 × 3개 중대)의 표를 AI가 읽기 쉬운 형태로 정리하고, 빠진 값·단위 혼동·범위를 벗어난 값이 없는지 점검 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `정리한 데이터 표, 데이터 점검 기록` 완성

### 📑 실습 서식 및 대조 점검 양식
실습가이드 지침에 따라 가상 데이터를 분석하고 프롬프트/점검표를 완성합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 정보 구조화 — 데이터를 읽기 쉬운 표로 정리하고 이상한 값과 비교 기준을 점검했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(정리한 데이터 표, 데이터 점검 기록)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 02.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_02_prompt = """[12행 생활 데이터 마크다운 표 정형화 및 무결성 검증 질의]
당신은 군 데이터 정제 및 무결성 검증 전문가입니다. 가상 한빛대대 3개 중대의 4주간 생활관 생활 데이터 원문을 AI가 분석하기 용이한 12행 표준 마크다운 표로 정형화해 주십시오.
또한 결측치, 만족도 점수 범위(1~5점 척도), 단위 일관성을 점검하는 데이터 무결성 점검 기록을 작성하고, 중대별 인원 차이(1중대 48명, 2중대 52명, 3중대 45명)를 고려한 1인당 지표 산출 계획을 수립해 주십시오."""

mission_02_result = """## 1. 정형 마크다운 데이터 표 (12행)
| 주차 | 중대 | 인원(명) | 체력단련실(회) | 싸지방(회) | PX(회) | 수면만족(5점) | 식단만족(5점) | 건의(건) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1주차 | 1중대 | 48 | 112 | 54 | 140 | 3.8 | 3.9 | 3 |
| 1주차 | 2중대 | 52 | 98 | 31 | 160 | 3.6 | 3.5 | 4 |
| 1주차 | 3중대 | 45 | 104 | 47 | 118 | 3.7 | 3.2 | 5 |
| 2주차 | 1중대 | 48 | 118 | 58 | 138 | 3.8 | 3.8 | 2 |
| 2주차 | 2중대 | 52 | 92 | 29 | 166 | 3.4 | 3.4 | 5 |
| 2주차 | 3중대 | 45 | 106 | 50 | 121 | 3.6 | 3.1 | 7 |
| 3주차 | 1중대 | 48 | 121 | 61 | 142 | 3.7 | 4.0 | 4 |
| 3주차 | 2중대 | 52 | 85 | 26 | 171 | 3.2 | 3.4 | 7 |
| 3주차 | 3중대 | 45 | 103 | 49 | 117 | 3.7 | 3.3 | 9 |
| 4주차 | 1중대 | 48 | 125 | 66 | 135 | 3.7 | 4.1 | 3 |
| 4주차 | 2중대 | 52 | 80 | 24 | 178 | 3.0 | 3.3 | 8 |
| 4주차 | 3중대 | 45 | 108 | 52 | 120 | 3.7 | 3.2 | 12 |

## 2. 데이터 무결성 점검 기록
- **결측치(빈 셀) 점검**: 108개 셀 중 결측치 ( 0 )건 확인
- **만족도 점수 범위(1.0~5.0) 점검**: 비정상 수치 ( 없음 )
- **단위 일관성 점검**: 이용량(회), 점수(5점 척도), 건의(건) 표기 일치 여부 확인 완료

## 3. 1인당 지표 산출 계획
- **분모 기준**: 1중대 ( 48 )명, 2중대 ( 52 )명, 3중대 ( 45 )명
- **산출 공식**: `1인당 4주 총 이용 횟수 = 4주간 총 이용 횟수 ÷ 중대 인원수`
- **적용 대상 항목**: 체력단련실, 싸지방, PX 이용 건수 (만족도 점수는 평균이므로 제외)

## 4. 도전 과제: 이상치 탐지 실험
- **변형 데이터 내용**: 
- **AI의 이상치 감지 여부 및 반응**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 02. 데이터 표 정리하고 점검하기", mission_02_prompt, mission_02_result)



In [ ]:
# [미션 02 실습 보조] 결측치 및 데이터 무결성 검증 (Pandas)
print('--- 결측치(Null) 점검 ---')
print(df.isnull().sum())

print('\n--- 만족도 점수 범위 점검 (1~5점 척도 여부) ---')
print(df[['수면만족', '식단만족']].describe().loc[['min', 'max']])

print('\n--- 1인당 계산용 중대별 정원 현황 ---')
display(df.groupby('중대')['인원(명)'].first().to_frame())